# Agentic AI 소개
## 소개 및 환경 설정

---

### 학습 목표
1. **uv** 기반 Python 3.11 가상환경 구성 (Windows 11 + CMD)
2. **로컬 Ollama + `qwen3:8b`** 를 기본 LLM 으로 연결하고, 클라우드 **Google(Gemini)** 과 한 줄로 전환
3. LLM과 AI Agent의 차이점 파악
4. Agent의 4가지 핵심 역량 이해 (Tool Use / Memory / Planning / Reasoning)
5. ReAct 패턴 이해 및 기본 실습

> 📦 **환경 설치·실행 명령**은 [`env_guides/M01_1_intro.md`](env_guides/M01_1_intro.md) 에 별도 정리되어 있습니다.
> 반복·공통 코드는 [`agentic_lib/`](agentic_lib) 라이브러리로 분리해 두었습니다.

---
## 1. uv 기반 환경 구성 (Windows 11 + CMD)

### uv란?
`uv`는 Astral이 만든 **초고속 Python 패키지 매니저**입니다.
기존 `pip` + `venv` + `pyenv`의 역할을 하나로 통합하며, Rust로 작성되어 pip 대비 **10-100배 빠릅니다**.

```
pip + venv + pyenv  →  uv  (All-in-one, 초고속)
```

> 본 강의는 **Windows 11 + CMD(`cmd.exe`) + Python 3.11** 을 고정 타깃으로 합니다.
> 모든 셸 명령은 CMD 문법으로 작성합니다.

### Step 1: uv 설치 (1회 — uv 설치 스크립트만 PowerShell 사용)
```bat
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

### Step 2: 가상환경 생성 및 패키지 설치 (프로젝트 루트에서)
```bat
uv venv --python 3.11
uv sync
```

### Step 3: Jupyter 커널 등록
```bat
uv run python -m ipykernel install --user --name=agentic-ai-venv --display-name "Agentic AI (uv)"
```

> 이 노트북의 **커널**을 `Agentic AI (uv)` 로 선택한 뒤 실행하세요.

In [1]:
# uv 버전 확인
import subprocess, sys

def check_tool(cmd: list, name: str):
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        version = result.stdout.strip() or result.stderr.strip()
        print(f"✓ {name}: {version}")
        return True
    except FileNotFoundError:
        print(f"✗ {name}: 설치되지 않음")
        return False

check_tool(["uv", "--version"], "uv")
check_tool([sys.executable, "--version"], "Python")
print(f"\n가상환경 경로: {sys.prefix}")
print(f"uv 환경 여부: {'✓ uv venv' if '.venv' in sys.prefix else '⚠ 일반 Python'}")

✓ uv: uv 0.11.8 (0e961dd9a 2026-04-27 x86_64-pc-windows-msvc)
✓ Python: Python 3.11.15

가상환경 경로: C:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\.venv
uv 환경 여부: ✓ uv venv


---
## 2. 패키지 설치 (uv)

### pyproject.toml 방식 (권장)
```toml
# pyproject.toml
[project]
name = "agentic-ai-tutorial"
version = "0.1.0"
requires-python = ">=3.11,<3.12"
dependencies = [
    "langchain>=0.3.0",
    "langchain-openai>=0.2.0",       # Ollama/llama.cpp(OpenAI 호환) 연결에 사용
    "langchain-google-genai>=2.0.0", # 클라우드 비교(Gemini)
    "langchain-anthropic>=0.3.0",
    "langgraph>=0.2.0",
    "python-dotenv>=1.0.0",
    "ipykernel>=6.0.0",
]
```

```bat
REM pyproject.toml 기반 한 번에 설치
uv sync
```

### 또는 직접 설치 (아래 셀)

In [2]:
# 패키지 설치 — 공통 헬퍼 utils.uv_install() 사용 (uv 우선, 실패 시 pip 폴백)
import sys, os
sys.path.insert(0, os.path.abspath(''))  # notebooks/ 를 import 경로에 추가
from utils import uv_install

# 이 노트북에서 쓰는 패키지 설치 (이미 uv sync 로 설치돼 있으면 빠르게 통과)
uv_install([
    'anthropic>=0.40.0', 'google-generativeai>=0.8.0', 'langchain>=0.3.0',
    'langchain-anthropic>=0.3.0', 'langchain-google-genai>=2.0.0',
    'langchain-openai>=0.2.0', 'langchain-community>=0.3.0',
    'openai>=1.50.0', 'python-dotenv>=1.0.0',
])


[uv] 설치 완료: ['anthropic>=0.40.0', 'google-generativeai>=0.8.0', 'langchain>=0.3.0', 'langchain-anthropic>=0.3.0', 'langchain-google-genai>=2.0.0', 'langchain-openai>=0.2.0', 'langchain-community>=0.3.0', 'openai>=1.50.0', 'python-dotenv>=1.0.0']


---
## 3. LLM 공급자 선택 — 기본은 **로컬 Ollama (`qwen3:8b`)**

본 강의의 **기본 LLM 은 로컬 Ollama + `qwen3:8b`** 입니다(무료·오프라인·재현성, 데이터 외부 미유출).
비교·대체용 **클라우드 무료 API** 로 **Google(Gemini)** 과 **NVIDIA build** 를 함께 제공하며,
`.env` 의 `LLM_PROVIDER` **한 줄만** 바꾸면 전환됩니다(코드 변경 없음).

| # | 공급자 | `LLM_PROVIDER` | 무료 | 권장 모델 | 비고 |
|---|--------|----------------|------|-----------|------|
| 1 | **Ollama (로컬)** ⭐기본 | `ollama` | ✅ | `qwen3:8b` | 로컬 GPU/CPU, OpenAI 호환 `/v1`, 네이티브 도구 호출 |
| 2 | **Google AI Studio** ⭐비교 | `google` | ✅ 무료티어 | `gemini-3.1-flash-lite` | 클라우드, 빠름 |
| 3 | **NVIDIA build** ⭐비교 | `nvidia` | ✅ 무료 크레딧 | `meta/llama-3.1-8b-instruct` | 클라우드, `ChatNVIDIA`(build.nvidia.com) |
| 4 | llama.cpp (로컬) | `llamacpp` | ✅ | GGUF | 프리빌트 휠(컴파일러 불필요) |
| 5 | Anthropic (API/OAuth) | `anthropic` | 유료 | `claude-haiku-4-5` | 안전성·긴 컨텍스트 |
| 6 | OpenAI | `openai` | 유료 | `gpt-4o-mini` | 범용 |

### Ollama 준비 (CMD)
```bat
winget install --id Ollama.Ollama -e   REM 설치(1회)
ollama serve                           REM 서버 실행(자동 실행 안 되면 수동)
ollama pull qwen3:8b                    REM 모델 다운로드(약 5GB, 1회)
curl.exe http://localhost:11434/api/tags   REM 동작 확인
```

### 무료 클라우드 API 키 발급 (선택)
- **Google AI Studio**: [aistudio.google.com](https://aistudio.google.com) → `Get API Key` → `.env` 에 `GOOGLE_API_KEY=AIza...`
- **NVIDIA build**: [build.nvidia.com](https://build.nvidia.com) 로그인 → 모델 페이지 → `Get API Key` → `.env` 에 `NVIDIA_API_KEY=nvapi-...`

> 모든 로컬/클라우드 공급자는 `utils.get_llm()` 으로 **동일하게** 사용합니다(§6-B 에서 두 무료 API 를 접속·테스트).
> 자세한 설치·문제해결은 [`env_guides/M01_1_intro.md`](env_guides/M01_1_intro.md) 참고.

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath(''))  # notebooks/ 디렉토리를 import 경로에 추가

import utils
utils.reload_env()  # .env 재로드 (LLM_PROVIDER 등 갱신) + 현재 공급자 상태 출력

from utils import (
    uv_install, get_llm, test_llm_connection,
    LLM_PROVIDER, GOOGLE_API_KEY, ANTHROPIC_API_KEY, ANTHROPIC_OAUTH_TOKEN,
    VLLM_BASE_URL, VLLM_MODEL, OPENAI_API_KEY,
    NVIDIA_API_KEY, NVIDIA_MODEL,   # NVIDIA build(무료 크레딧) 접속용
    print_provider_status,
)

# 반복/공통 코드는 agentic_lib 라이브러리로 분리해 두었습니다(이 노트북 전반에서 사용).
from agentic_lib import bootstrap, tools, memory, planning, react
from agentic_lib.bootstrap import to_text  # 공급자 무관 응답 정규화(<think> 제거 포함)

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


---
## 4. 인증/설정 (`.env`)

`notebooks/.env.example` 를 복사해 `notebooks/.env` 를 만들고, 사용하는 공급자만 채웁니다.

```bat
cd notebooks
copy .env.example .env
```

```ini
# notebooks/.env  — 기본은 로컬 Ollama
LLM_PROVIDER=ollama
OLLAMA_BASE_URL=http://localhost:11434/v1
OLLAMA_MODEL=qwen3:8b

# 클라우드 무료 API 와 비교하려면 위 LLM_PROVIDER 를 google/nvidia 로 바꾸고 아래 키를 채움
GOOGLE_API_KEY=AIzaSy...
NVIDIA_API_KEY=nvapi-...
NVIDIA_MODEL=meta/llama-3.1-8b-instruct   # (선택) 기본값
```

> **Google API Key 발급(무료)**: [aistudio.google.com](https://aistudio.google.com) → `Get API Key` → `Create API key`
> → `.env` 에 `GOOGLE_API_KEY=...` 저장 후 `LLM_PROVIDER=google`
>
> **NVIDIA API Key 발급(무료 크레딧)**: [build.nvidia.com](https://build.nvidia.com) 로그인 → 모델 페이지 → `Get API Key`
> → `.env` 에 `NVIDIA_API_KEY=nvapi-...` 저장 후 `LLM_PROVIDER=nvidia`

In [4]:
# .env 파일이 있으면 utils.py import 시 자동으로 load_dotenv()가 호출됩니다.
# 추가로 확인하고 싶다면 아래를 실행하세요.
import os
print('LLM_PROVIDER:', os.getenv('LLM_PROVIDER', '(미설정 — 기본값: google)'))


LLM_PROVIDER: nvidia


---
## 5. 공급자별 연결 상세 설정

In [5]:
# ============================================================
# [참고] 공급자별 연결 — utils.py 의 get_llm() 으로 통합
# ============================================================
# get_llm() 이 아래 설정을 자동 처리합니다(.env 의 LLM_PROVIDER 로 선택):
#
#   "ollama"   → ChatOpenAI(base_url=OLLAMA_BASE_URL, model="qwen3:8b")   ⭐기본
#   "google"   → ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")    ⭐비교
#   "llamacpp" → ChatOpenAI(base_url=LLAMACPP_BASE_URL)
#   "anthropic"→ ChatAnthropic(model="claude-haiku-4-5-20251001")
#   "openai"   → ChatOpenAI(model="gpt-4o-mini")
#
# 로컬(ollama/llamacpp)도 모두 OpenAI 호환 /v1 이라 클라우드와 동일 인터페이스로 씁니다.
print("공급자별 연결은 utils.get_llm() 으로 통합됩니다.")
print(f"현재 선택된 공급자: {LLM_PROVIDER}")

공급자별 연결은 utils.get_llm() 으로 통합됩니다.
현재 선택된 공급자: nvidia


---
## 6. 통합 LLM 인터페이스 (모든 공급자 공통)

이후 실습에서는 `get_llm()` 함수 하나로 모든 공급자를 동일하게 사용합니다.

In [6]:
# get_llm()은 utils.py에 정의되어 있습니다. 이미 import되었습니다.
# 다른 공급자로 테스트:
# llm_google    = get_llm('google')
# llm_anthropic = get_llm('anthropic')
# llm_vllm      = get_llm('vllm')
print('get_llm() 사용 준비 완료. llm = get_llm() 또는 test_llm_connection()을 호출하세요.')


get_llm() 사용 준비 완료. llm = get_llm() 또는 test_llm_connection()을 호출하세요.


In [7]:
llm = test_llm_connection()

LLM 연결 성공 [nvidia]: 1+1은 2입니다.


---
## 6-B. 무료 클라우드 LLM API 테스트 — Google & NVIDIA build

기본 LLM 은 로컬 Ollama 지만, 여기서는 **무료로 쓸 수 있는 두 클라우드 API** 를 함께
`utils.get_llm()` 으로 접속·테스트합니다. 로컬 자원이 부족하거나 더 큰 모델을 잠깐 써 보고 싶을 때 유용합니다.

| 서비스 | 무료 | 키 발급 | 접속(공통 팩토리) | 예시 모델 |
|---|---|---|---|---|
| **Google AI Studio** | ✅ 무료 티어 | [aistudio.google.com](https://aistudio.google.com) | `utils.get_llm('google')` | `gemini-3.1-flash-lite` |
| **NVIDIA build** | ✅ 무료 크레딧 | [build.nvidia.com](https://build.nvidia.com) | `utils.get_llm('nvidia')` | `meta/llama-3.1-8b-instruct` |

- `.env` 에 `GOOGLE_API_KEY`, `NVIDIA_API_KEY` 중 **있는 것만** 채우면 됩니다. 키가 없는 서비스는 오류 없이 **발급 안내만** 출력합니다.
- 두 서비스 모두 `utils.get_llm()` 이 알맞은 커넥터(`ChatGoogleGenerativeAI` / `ChatNVIDIA`)를 돌려주므로, 로컬/구글과 **동일한 `.invoke()` 인터페이스**로 다룹니다.
- 스트리밍 비교 등 더 자세한 예제는 [`M02_0_free_llm_api.ipynb`](M02_0_free_llm_api.ipynb) 참고.

In [8]:
# === 무료 클라우드 LLM API 접속·테스트 (Google · NVIDIA build) ===
# 기본 LLM(ollama) 과 별개로, 무료 클라우드 API 두 곳을 utils.get_llm() 으로 바로 호출해 봅니다.
from langchain_core.messages import HumanMessage, SystemMessage

# 두 서비스의 LangChain 커넥터 설치(이미 있으면 빠르게 통과)
uv_install(['langchain-google-genai', 'langchain-nvidia-ai-endpoints'])

# 키 설정 상태(.env 에서 로드)
print('Google AI Studio 키:', '설정됨' if GOOGLE_API_KEY else '미설정 — aistudio.google.com')
print('NVIDIA build 키   :', '설정됨' if NVIDIA_API_KEY else '미설정 — build.nvidia.com')
print('-' * 60)

# 공통 테스트 프롬프트
CLOUD_PROMPT = [
    SystemMessage(content='간결하게 한국어로 답하세요.'),
    HumanMessage(content='에이전틱(Agentic) AI를 한 문장으로 설명해줘.'),
]

def test_cloud_api(provider: str, label: str, api_key: str):
    """무료 클라우드 공급자 하나를 접속·호출 테스트한다(키 없으면 안내만 출력)."""
    if not api_key:
        print(f'[{label}] 키 미설정 — 발급 후 .env 에 추가하면 테스트됩니다.')
        return
    try:
        # google → ChatGoogleGenerativeAI / nvidia → ChatNVIDIA (동일 인터페이스)
        model = get_llm(provider)
        resp = model.invoke(CLOUD_PROMPT)
        print(f'[{label}] {to_text(resp.content)}')   # 공급자 무관 응답 정규화
    except Exception as e:
        print(f'[{label}] 호출 실패: {type(e).__name__}: {e}')

# 각 서비스 접속 테스트 (기본 공급자 ollama 와 무관하게 항상 실행)
test_cloud_api('google', 'Google Gemini', GOOGLE_API_KEY)
test_cloud_api('nvidia', f'NVIDIA {NVIDIA_MODEL}', NVIDIA_API_KEY)

[uv] 설치 완료: ['langchain-google-genai', 'langchain-nvidia-ai-endpoints']
Google AI Studio 키: 설정됨
NVIDIA build 키   : 설정됨
------------------------------------------------------------


[Google Gemini] 에이전틱 AI는 단순히 명령을 수행하는 것을 넘어, 스스로 목표를 설정하고 계획을 세워 자율적으로 문제를 해결하는 지능형 시스템입니다.


[NVIDIA meta/llama-3.1-8b-instruct] 에이전틱(Agentic) AI는 사용자의 의도와 목표를 이해하고 이를 달성하기 위한 행동을 취하는 인공지능을 말합니다.


---
## 7. LLM vs AI Agent: 무엇이 다른가?

### 일반 LLM
- **역할:** 입력 텍스트에 대한 텍스트 출력 생성
- **한계:** 학습 데이터 기준의 답변, 실시간 정보 없음, 외부 도구 사용 불가
- **비유:** 아무리 박식한 교수님도 인터넷 없이 대화만 하는 상황

### AI Agent
- **역할:** 목표 달성을 위해 스스로 계획하고, 도구를 사용하며, 결과를 평가
- **강점:** 실시간 정보 검색, 코드 실행, 외부 API 호출, 다단계 계획 수립
- **비유:** 비서처럼 직접 인터넷 검색, 일정 조율, 문서 작성까지 수행

```
[LLM 방식]
사용자 질문 → LLM → 텍스트 답변

[Agent 방식]
사용자 목표 → Agent ┌→ 계획 수립
                    ├→ 도구 선택 및 실행
                    ├→ 결과 관찰
                    ├→ 재계획 (필요시)
                    └→ 최종 답변
```

---
## 8. Agent의 4가지 핵심 역량

| 역량 | 비유 | 역할 |
|------|------|------|
| **Tool Use** | 손 | 외부 API 호출, 웹 검색, 코드 실행 |
| **Memory** | 기억 | 과거 대화, 장기 지식 저장 및 활용 |
| **Planning** | 전략 | 복잡한 문제를 단계별로 분해 |
| **Reasoning** | 지능 | 실시간 피드백으로 논리적 사고 및 결정 |

In [9]:
# === Agent의 4가지 핵심 역량 — 실제 LLM 데모 ===
# 공통 도구는 agentic_lib.tools 에서 가져오고, 응답은 to_text() 로 깔끔하게 출력합니다.
# (Gemini 는 content 가 list, qwen3 는 <think> 가 섞여 나오므로 to_text 로 통일)
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

if llm is None:
    print('LLM 이 연결되지 않았습니다.')
else:
    # ── 1. Tool Use ──────────────────────────────────────────────────────
    print('━' * 60)
    print('1. Tool Use: LLM 이 외부 도구를 직접 호출합니다')
    print('━' * 60)

    # calculator/get_current_time 는 라이브러리에서 제공(중복 정의 제거)
    available_tools = [tools.calculator, tools.get_current_time]
    # bootstrap.bind_tools: 단일 도구 서버(NVIDIA build)에는 parallel_tool_calls=False 유도
    llm_with_tools = bootstrap.bind_tools(llm, available_tools)

    # 도구 이름 → 도구 객체 매핑(레지스트리). LLM 이 고른 이름으로 알맞은 도구를 찾아 실행한다.
    tool_registry = {t.name: t for t in available_tools}

    question = '2의 10제곱이 얼마야?'
    print(f'[질문] {question}')
    resp = llm_with_tools.invoke([HumanMessage(content=question)])
    # 단일 도구 서버면 tool_calls 를 첫 1개로 줄여 이력이 도구 하나만 담게 한다(그 외 원본 유지)
    resp = bootstrap.cap_tool_calls(resp)

    if resp.tool_calls:
        tc = resp.tool_calls[0]
        tool_name = tc['name']
        print(f'[LLM 결정] 도구 선택: {tool_name}({tc["args"]})')
        # 하드코딩 대신 이름으로 알맞은 도구를 조회해 실행
        selected_tool = tool_registry.get(tool_name)
        if selected_tool is None:
            tool_result = '알 수 없는 도구: ' + tool_name
        else:
            tool_result = selected_tool.invoke(tc['args'])
        print(f'[도구 실행] {tool_result}')
        final = llm_with_tools.invoke([
            HumanMessage(content=question), resp,
            ToolMessage(content=str(tool_result), tool_call_id=tc['id'])
        ])
        print(f'[최종 답변] {to_text(final.content)}')  # to_text 로 list/think 정규화
    else:
        print(f'[LLM 답변(도구 미사용)] {to_text(resp.content)}')

    # ── 2. Memory ─────────────────────────────────────────────────────────
    print('\n' + '━' * 60)
    print('2. Memory: 이전 대화를 기억합니다')
    print('━' * 60)

    history = [SystemMessage(content='당신은 사용자 정보를 기억하는 AI 어시스턴트입니다.')]
    for user_input in ['제 이름은 김철수이고 서울에 살아요.', '제 이름이 뭐고 어디 산다고 했죠?']:
        history.append(HumanMessage(content=user_input))
        print(f'[사용자] {user_input}')
        reply = llm.invoke(history)
        history.append(reply)
        print(f'[에이전트] {to_text(reply.content)[:120]}')

    # ── 3. Planning ───────────────────────────────────────────────────────
    print('\n' + '━' * 60)
    print('3. Planning: LLM 이 복잡한 작업을 단계별로 분해합니다')
    print('━' * 60)

    plan_resp = llm.invoke([
        SystemMessage(content='요청을 번호가 붙은 구체적인 실행 단계로 분해하세요. 각 단계를 한 줄로.'),
        HumanMessage(content='AI 트렌드 분석 보고서 작성 계획을 세워줘.')
    ])
    print(f'[계획]\n{to_text(plan_resp.content)}')

    # ── 4. Reasoning ──────────────────────────────────────────────────────
    print('\n' + '━' * 60)
    print('4. Reasoning: 단계별 논리적 추론 (Chain of Thought)')
    print('━' * 60)

    cot_resp = llm.invoke([
        SystemMessage(content=(
            '문제를 단계별로 논리적으로 추론하세요. '
            '각 단계를 "단계 N:" 형식으로 명확히 설명한 뒤 최종 답을 주세요.'
        )),
        HumanMessage(content=(
            '사과가 24개 있고 6명이 똑같이 나눠 가진 후, '
            '각자 자신의 몫에서 절반을 먹었다면 남은 사과는 총 몇 개?'
        ))
    ])
    print(f'[추론]\n{to_text(cot_resp.content)}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Tool Use: LLM 이 외부 도구를 직접 호출합니다
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[질문] 2의 10제곱이 얼마야?


[LLM 결정] 도구 선택: calculator({'expression': '2 ** 10'})
[도구 실행] 2 ** 10 = 1024


[최종 답변] 2의 10제곱은 1024입니다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2. Memory: 이전 대화를 기억합니다
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[사용자] 제 이름은 김철수이고 서울에 살아요.


[에이전트] 안녕하세요 김철수님! 서울에 살고 계신 것 같아요. 어떻게 지내시나요?
[사용자] 제 이름이 뭐고 어디 산다고 했죠?


[에이전트] 네, 김철수님이라고 하셨고, 서울에 살고 계신다고 말씀하셨습니다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
3. Planning: LLM 이 복잡한 작업을 단계별로 분해합니다
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


[계획]
1. **보고서 목적과 범위 정의**: AI 트렌드 분석 보고서의 목적, 범위, 대상audience를 명확히 정의합니다.
2. **AI 트렌드 수집**: AI 트렌드 분석을 위한 데이터 수집을 위해 관련 기사, 연구-paper, 보고서, 컨퍼런스 자료 등 다양한 소스를 수집합니다.
3. **트렌드 분류**: 수집한 데이터를 AI 트렌드의 유형(예: 자연어 처리, computer vision, 강화 학습 등)으로 분류합니다.
4. **트렌드 분석**: 각 트렌드의 특징, 장점, 단점, 예상 영향, 예상 시기 등을 분석합니다.
5. **트렌드 예측**: 분석 결과를 기반으로 AI 트렌드의 미래 예측을 합니다.
6. **보고서 구조 설계**: 보고서의 구조를 설계합니다. 일반적으로 Executive Summary, Introduction, Methodology, Analysis, Conclusion, Recommendation 등으로 구성됩니다.
7. **Executive Summary 작성**: 보고서의 주요 내용을 요약하여 Executive Summary를 작성합니다.
8. **Introduction 작성**: 보고서의 목적, 범위, 대상audience를 설명하여 Introduction을 작성합니다.
9. **Methodology 작성**: 데이터 수집, 분류, 분석 방법을 설명하여 Methodology를 작성합니다.
10. **Analysis 작성**: 각 트렌드의 특징, 장점, 단점, 예상 영향, 예상 시기 등을 분석하여 Analysis를 작성합니다.
11. **Conclusion 작성**: 분석 결과를 요약하여 Conclusion을 작성합니다.
12. **Recommendation 작성**: 분석 결과를 기반으로 AI 트렌드에 대한 추천을 작성합니다.
13. **보고서 작성**: 각 섹션을 작성하여 보고서를 완성합니다.
14. **보고서 리뷰**: 보고서를 리뷰하여 오류를 수정하고 내용을 보완합니다.
15. **보고서 최종화**: 보고서를 최종화하

[추론]
단계 1: 사과가 24개 있고 6명이 똑같이 나눠 가진 경우, 각 사람의 몫은 24 / 6 = 4개입니다.

단계 2: 각 사람의 몫에서 절반을 먹었다면, 각 사람의 남은 사과는 4 / 2 = 2개입니다.

단계 3: 6명이 모두 남은 사과를 가지고 있으므로, 총 남은 사과는 6 * 2 = 12개입니다.

최종 답: 남은 사과는 총 12개입니다.


---
## 9. ReAct 패턴 (Reason + Act)

ReAct는 에이전트의 핵심 사고 루프입니다:

```
목표 입력
  ↓
Thought: 현재 상황 분석, 다음 행동 결정
  ↓
Action: 도구 선택 및 실행
  ↓
Observation: 도구 실행 결과 확인
  ↓
(반복 또는 종료)
  ↓
Final Answer: 최종 답변 제시
```

**논문:** Yao et al., "ReAct: Synergizing Reasoning and Acting in Language Models" (ICLR 2023)

In [10]:
# ReAct 패턴 직접 구현 (LLM 없이 로직 이해용) — agentic_lib.react 사용
# SimpleReActAgent / 규칙기반 도구(PLAIN_TOOLS) 는 라이브러리로 분리되어 있습니다.
agent = react.SimpleReActAgent()          # 기본 도구 = react.PLAIN_TOOLS
TOOLS = react.PLAIN_TOOLS                 # 아래 Planning 데모에서 재사용

print('=' * 60); print('테스트 1: 현재 시간'); print('=' * 60)
agent.run('지금 몇 시야?')
print(); print('=' * 60); print('테스트 2: 수학 계산'); print('=' * 60)
agent.run('2 ** 10 계산해줘')

테스트 1: 현재 시간
[목표] 지금 몇 시야?
--------------------------------------------------
[Thought 1] 사용할 도구: get_current_time
[Action 1] get_current_time('')
[Observation 1] 2026년 07월 25일 15:27:09

[Final Answer] 2026년 07월 25일 15:27:09

테스트 2: 수학 계산
[목표] 2 ** 10 계산해줘
--------------------------------------------------
[Thought 1] 사용할 도구: calculator
[Action 1] calculator('2 ** 10')
[Observation 1] 2 ** 10 = 1024

[Final Answer] 2 ** 10 = 1024


'2 ** 10 = 1024'

---
## 10. 선택한 LLM으로 실제 ReAct 에이전트 구현

각 공급자의 Tool Calling API를 통일된 인터페이스로 사용합니다.

In [11]:
from langchain.agents import create_agent

# LangChain 도구는 agentic_lib.tools 의 공통 도구를 사용(중복 정의 제거)
lc_tools = [tools.calculator, tools.get_current_time, tools.search_web]

system_prompt = (
    "당신은 도구를 사용하는 유능한 AI 에이전트입니다. "
    "사용자의 요청을 분석하고 적절한 도구를 선택해 정확한 답변을 제공하세요. "
    "항상 한국어로 답변하세요."
)

def run_react_agent(question: str):
    """선택된 LLM 공급자로 ReAct 에이전트(create_agent)를 실행한다."""
    if llm is None:
        print("LLM이 연결되지 않았습니다. 섹션 6을 먼저 실행하세요.")
        return
    try:
        agent = create_agent(llm, lc_tools, system_prompt=system_prompt)
        result = agent.invoke({"messages": [("human", question)]})
        print(f"\n최종 답변: {to_text(result['messages'][-1].content)}")
        return result
    except Exception as e:
        print(f"에이전트 오류: {type(e).__name__}: {e}")

print(f"=== ReAct Agent ({LLM_PROVIDER}) ===")
run_react_agent("지금 몇 시인지 알려주고, 2의 10제곱도 계산해줘")

=== ReAct Agent (nvidia) ===


에이전트 오류: Exception: [500] {'message': 'Failed to generate completions: Failed to apply prompt template: invalid operation: This model only supports single tool-calls at once! (in tool_use:95)', 'type': 'Internal Server Error', 'code': 500}
{'error': {'message': 'Failed to generate completions: Failed to apply prompt template: invalid operation: This model only supports single tool-calls at once! (in tool_use:95)', 'type': 'Internal Server Error', 'code': 500}, 'status': 500}


---
## 11. 공급자별 직접 호출 비교

LangChain 없이 각 공급자 SDK를 직접 사용하는 방법도 알아봅니다.

In [12]:
def call_llm_direct(prompt: str, provider: str = None) -> str:
    """
    공급자별 SDK를 직접 호출합니다 (LangChain 미사용).
    각 SDK의 차이를 이해하기 위한 참고용 코드입니다.
    (로컬 ollama/llamacpp 는 OpenAI SDK 로 base_url 만 바꿔 호출)
    """
    p = provider or LLM_PROVIDER

    # ── Google AI Studio ────────────────────────────────────
    if p == "google":
        from google import genai
        client = genai.Client(api_key=GOOGLE_API_KEY)
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt,
        )
        return response.text

    # ── Anthropic API Key ────────────────────────────────────
    elif p == "anthropic":
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

    # ── Anthropic OAuth ──────────────────────────────────────
    elif p == "anthropic_oauth":
        import anthropic
        client = anthropic.Anthropic(auth_token=ANTHROPIC_OAUTH_TOKEN)
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

    # ── vLLM (OpenAI 호환) ───────────────────────────────────
    elif p == "vllm":
        from openai import OpenAI
        client = OpenAI(base_url=VLLM_BASE_URL, api_key="EMPTY")
        response = client.chat.completions.create(
            model=VLLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=256
        )
        return response.choices[0].message.content

    # ── OpenAI ──────────────────────────────────────────────
    elif p == "openai":
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=256
        )
        return response.choices[0].message.content

    # ── Ollama / llama.cpp (로컬, OpenAI 호환) ───────────────
    elif p in ("ollama", "llamacpp"):
        from openai import OpenAI
        # 로컬 서버 주소·모델은 utils(.env) 에서 가져온다
        base_url = utils.OLLAMA_BASE_URL if p == "ollama" else utils.LLAMACPP_BASE_URL
        model    = utils.OLLAMA_MODEL   if p == "ollama" else utils.LLAMACPP_MODEL
        # api_key 는 형식상 필요(로컬 서버는 값을 검증하지 않음)
        client = OpenAI(base_url=base_url, api_key="ollama")
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt + " /no_think"}],  # qwen3 사고 생략
            max_tokens=512,  # /no_think 여도 최소 토큰이 부족하면 빈 응답이 나므로 넉넉히
        )
        # qwen3 등 사고 모델의 <think> 블록을 제거해 깔끔한 문자열로 반환
        return bootstrap.strip_think(response.choices[0].message.content)

    return f"지원하지 않는 공급자: {p}"


# 직접 호출 테스트
test_prompt = "에이전틱 AI를 한 문장으로 설명해주세요."
try:
    answer = call_llm_direct(test_prompt)
    print(f"[{LLM_PROVIDER}] 직접 호출 결과:")
    print(f"  {answer}")
except Exception as e:
    print(f"호출 실패: {type(e).__name__}: {e}")

[nvidia] 직접 호출 결과:
  지원하지 않는 공급자: nvidia


---
## 12. 메모리(Memory) 개념 실습

- **단기 기억(Short-term):** 현재 대화의 맥락 유지
- **장기 기억(Long-term):** 외부 저장소에 영구 저장

In [13]:
# 단기/장기 메모리 — agentic_lib.memory 사용(클래스 구현은 라이브러리에 분리)
short = memory.ConversationMemory(max_messages=5)
long  = memory.SimpleVectorMemory()

short.add("user", "제 이름은 김철수입니다.")
short.add("assistant", "안녕하세요, 김철수님!")
short.add("user", "저는 AI 개발자입니다.")

print("=== 단기 기억 (슬라이딩 윈도우) ===")
for m in short.get():
    print(f"  [{m['role']}] {m['content']}")

long.save("사용자_프로필", "이름: 김철수, 직업: AI 개발자")
long.save("프로젝트_A", "LangGraph 기반 에이전트 시스템 구현")

print("\n=== 장기 기억 검색 ('김철수') ===")
for r in long.search("김철수"):
    print(f"  [{r['key']}] {r['value']}")

=== 단기 기억 (슬라이딩 윈도우) ===
  [user] 제 이름은 김철수입니다.
  [assistant] 안녕하세요, 김철수님!
  [user] 저는 AI 개발자입니다.
[저장] '사용자_프로필'
[저장] '프로젝트_A'

=== 장기 기억 검색 ('김철수') ===
  [사용자_프로필] 이름: 김철수, 직업: AI 개발자


---
## 13. 플래닝(Planning) 개념 실습

In [14]:
# 작업 계획·실행 — agentic_lib.planning 사용(Task/TaskStatus/TaskPlanner 는 라이브러리)
planner = planning.TaskPlanner()
t1 = planner.add("AI 트렌드 검색", tool="search_web")
t2 = planner.add("현재 날짜 확인", tool="get_current_time")
t3 = planner.add("보고서 구조 설계", depends_on=[t1.id, t2.id])
t4 = planner.add("보고서 초안 작성", depends_on=[t3.id])
t5 = planner.add("검토 및 완료",    depends_on=[t4.id])

planner.show()
planner.execute(TOOLS)   # TOOLS = react.PLAIN_TOOLS (앞 셀에서 정의)
print()
planner.show()

=== 실행 계획 ===
  [대기  ] Task 1: AI 트렌드 검색
  [대기  ] Task 2: 현재 날짜 확인
  [대기  ] Task 3: 보고서 구조 설계 (의존: [1, 2])
  [대기  ] Task 4: 보고서 초안 작성 (의존: [3])
  [대기  ] Task 5: 검토 및 완료 (의존: [4])

=== 실행 ===
  Task 1: AI 트렌드 검색
    → 'AI 트렌드 검색' 검색 결과: 관련 정보 3건 발견 (시뮬레이션)
  Task 2: 현재 날짜 확인
    → 2026년 07월 25일 15:27:12
  Task 3: 보고서 구조 설계
    → 완료
  Task 4: 보고서 초안 작성
    → 완료
  Task 5: 검토 및 완료
    → 완료

=== 실행 계획 ===
  [완료  ] Task 1: AI 트렌드 검색
  [완료  ] Task 2: 현재 날짜 확인
  [완료  ] Task 3: 보고서 구조 설계 (의존: [1, 2])
  [완료  ] Task 4: 보고서 초안 작성 (의존: [3])
  [완료  ] Task 5: 검토 및 완료 (의존: [4])


---
## 14. 정리 및 다음 모듈 예고

### 이번 시간 핵심 요약

| 주제 | 핵심 내용 |
|------|----------|
| **uv** | pip 대비 10-100배 빠른 패키지 매니저. `uv venv --python 3.11` + `uv sync` |
| **기본 LLM** | **로컬 Ollama + `qwen3:8b`** (무료·오프라인). 클라우드 `google`/`nvidia` 는 한 줄 전환 비교 |
| **무료 클라우드 API** | **Google AI Studio(Gemini)** · **NVIDIA build(`ChatNVIDIA`)** — `utils.get_llm('google'/'nvidia')` 로 접속·테스트(§6-B) |
| **LLM vs Agent** | LLM = 텍스트 생성기 / Agent = 자율 실행 시스템 |
| **4가지 역량** | Tool Use(손) · Memory(기억) · Planning(전략) · Reasoning(지능) |
| **ReAct** | Thought → Action → Observation 루프 반복 |
| **라이브러리** | 반복 코드는 `agentic_lib`(tools·memory·planning·react·bootstrap)로 분리 |

### LLM 공급자 선택 가이드
```
무료로 로컬에서 (기본·권장)   → Ollama + qwen3:8b   (LLM_PROVIDER=ollama)
클라우드와 빠르게 비교        → Google Gemini       (LLM_PROVIDER=google)
클라우드 무료 크레딧          → NVIDIA build        (LLM_PROVIDER=nvidia)
컴파일러 없이 GGUF 로컬 서빙  → llama.cpp           (LLM_PROVIDER=llamacpp)
안전성·긴 컨텍스트           → Anthropic Claude
```

### 다음 주차: 모듈 1 - 연결 (Connectivity & Skills)
- **MCP(Model Context Protocol):** 에이전트와 외부 서비스 연결 표준
- **A2A(Agent-to-Agent):** 에이전트 간 협업 아키텍처
- **로컬 LLM 서빙:** Ollama / llama.cpp 로 도구 연동

---
### 참고 자료
- uv 공식 문서: https://docs.astral.sh/uv/
- Ollama: https://ollama.com  ·  qwen3: https://ollama.com/library/qwen3
- Google AI Studio: https://aistudio.google.com
- NVIDIA build: https://build.nvidia.com  ·  LangChain 커넥터: `langchain-nvidia-ai-endpoints`
- Anthropic API 문서: https://docs.anthropic.com
- ReAct 논문: https://arxiv.org/abs/2210.03629